# 2 - Pre-processamento

Este notebook transforma a exploracao inicial em um pipeline de preparo dos dados para modelagem. A proposta aqui e ser mais criterioso do que a versao inicial: primeiro identificamos colunas inviaveis, depois diferenciamos sinais analogicos de sinais de estado, tratamos faltantes de forma coerente com cada grupo e por fim geramos versoes escaladas e persistidas para as proximas etapas.


## 1. Ambiente

Bibliotecas usadas:

- `pandas`, `numpy`
- `matplotlib`, `seaborn`
- `scikit-learn`
- `torch` para a etapa posterior de sequencias LSTM

Se necessario, habilite a instalacao abaixo:

```python
# %pip install pyarrow torch scikit-learn seaborn
```


In [5]:
from __future__ import annotations

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

try:
    #torch so sera usado nas etapas seguintes, mas ja deixamos o ambiente conferido
    import torch
except ModuleNotFoundError:
    torch = None

#padroniza a aparencia dos graficos do notebook
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 16

#printa versoes para facilitar reproducao do pipeline
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"torch: {torch.__version__}" if torch is not None else "torch: nao instalado")


pandas: 2.3.3
numpy: 1.26.4
torch: 2.2.2


## 2. Carga da instancia local

O notebook usa o clone local em `3W/dataset/`. Por padrao, ele carrega a primeira instancia encontrada, mas voce pode substituir manualmente o arquivo escolhido se quiser trabalhar com outro evento ou outro poco.


In [6]:
PROJECT_ROOT = Path.cwd()
LOCAL_DATASET_DIR = PROJECT_ROOT / "3W" / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)


def resolve_parquet_file() -> Path:
    #confere se a pasta do dataset existe na estrutura local esperada
    if not LOCAL_DATASET_DIR.exists():
        raise FileNotFoundError(
            f"Pasta nao encontrada: {LOCAL_DATASET_DIR}. Esperado um clone local em 3W/dataset/."
        )

    #procura todos os parquet e escolhe a primeira instancia disponivel
    local_files = sorted(LOCAL_DATASET_DIR.rglob("*.parquet"))
    if not local_files:
        raise FileNotFoundError(f"Nenhum arquivo .parquet foi encontrado em {LOCAL_DATASET_DIR}.")

    print(f"Usando arquivo local: {local_files[0]}")
    return local_files[0]


parquet_file = resolve_parquet_file()
df = pd.read_parquet(parquet_file)

#padroniza o tempo no indice para viabilizar a interpolacao temporal
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.set_index("timestamp")
else:
    df.index = pd.to_datetime(df.index, errors="coerce")

df = df.sort_index()
df.index.name = "timestamp"

print(f"Shape inicial: {df.shape}")
print(f"Arquivo analisado: {parquet_file}")
print(f"Periodo: {df.index.min()} -> {df.index.max()}")


Usando arquivo local: /Users/tiagoriosdarocha/Desktop/lstm-w3/3W/dataset/0/WELL-00001_20170201010207.parquet
Shape inicial: (21474, 29)
Arquivo analisado: /Users/tiagoriosdarocha/Desktop/lstm-w3/3W/dataset/0/WELL-00001_20170201010207.parquet
Periodo: 2017-02-01 01:02:07 -> 2017-02-01 07:00:00


## 3. Diagnostico para limpeza

Antes de aplicar qualquer imputacao ou scaler, precisamos entender quais colunas fazem sentido manter. Em series industriais isso e fundamental: uma coluna totalmente vazia nao deve ir para a modelagem, e uma coluna binaria nao deve receber o mesmo tratamento de uma pressao ou temperatura.


In [7]:
#mede faltantes e variabilidade para orientar a limpeza
null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
constant_ratio = df.nunique(dropna=True).sort_values()

diagnostic_df = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
        "null_pct": null_pct,
        "nunique": constant_ratio,
    }
).sort_values(["null_pct", "nunique"], ascending=[False, True])

diagnostic_df


,dtype,null_count,null_pct,nunique
ABER-CKGL,float64,21474,100.000000,0
ABER-CKP,float64,21474,100.000000,0
P-JUS-BS,float64,21474,100.000000,0
P-JUS-CKP,float64,21474,100.000000,0
P-MON-CKGL,float64,21474,100.000000,0
P-MON-SDV-P,float64,21474,100.000000,0
PT-P,float64,21474,100.000000,0
QBS,float64,21474,100.000000,0
T-MON-CKP,float64,21474,100.000000,0
class,Int16,3600,16.764459,1


In [8]:
#identifica grupos de colunas com regras de tratamento diferentes
all_null_columns = diagnostic_df.index[diagnostic_df["null_pct"] == 100].tolist()
constant_columns = diagnostic_df.index[diagnostic_df["nunique"] <= 1].tolist()
label_columns = [col for col in ["class", "state", "label", "id"] if col in df.columns]
state_like_columns = [col for col in df.columns if col.startswith("ESTADO-") or col.startswith("ABER-")]

analog_candidate_columns = [
    col for col in df.select_dtypes(include=[np.number]).columns
    if col not in set(label_columns + state_like_columns)
]

print("Colunas 100% nulas:", all_null_columns)
print("Colunas constantes ou quase triviais:", constant_columns)
print("Colunas de rotulo/apoio:", label_columns)
print("Colunas de estado/binarias:", state_like_columns)
print("Candidatas analogicas:", analog_candidate_columns)


Colunas 100% nulas: ['ABER-CKGL', 'ABER-CKP', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP']
Colunas constantes ou quase triviais: ['ABER-CKGL', 'ABER-CKP', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP', 'class', 'state', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-PDG', 'QGL', 'T-PDG']
Colunas de rotulo/apoio: ['class', 'state']
Colunas de estado/binarias: ['ABER-CKGL', 'ABER-CKP', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO']
Candidatas analogicas: ['P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT']


## 4. Estrategia de limpeza

Regras adotadas neste notebook:

1. Colunas 100% nulas sao descartadas imediatamente.
2. `class` e `state` nao entram como features neste momento.
3. Colunas de estado/binarias sao preservadas como estao, sem interpolacao continua.
4. Colunas analogicas recebem imputacao temporal.
5. Colunas constantes sao removidas das features, porque costumam agregar pouco sinal e ainda podem atrapalhar o escalonamento.


In [9]:
#cria uma copia de trabalho para a etapa de limpeza
feature_df = df.copy()

#descarta colunas totalmente vazias, pois elas nao agregam informacao
drop_all_null = all_null_columns.copy()
feature_df = feature_df.drop(columns=drop_all_null, errors="ignore")

#remove labels e colunas auxiliares para evitar vazamento nas features
feature_df = feature_df.drop(columns=label_columns, errors="ignore")

#separa sinais de estado dos sinais analogicos
state_columns = [col for col in state_like_columns if col in feature_df.columns]
analog_columns = [
    col for col in feature_df.select_dtypes(include=[np.number]).columns
    if col not in state_columns
]

#marca colunas sem variacao util antes da limpeza final
constant_feature_columns = [
    col for col in feature_df.columns
    if feature_df[col].nunique(dropna=True) <= 1
]

print("Colunas removidas por 100% nulas:", drop_all_null)
print("Colunas removidas por serem labels/apoio:", label_columns)
print("Colunas constantes ainda presentes antes da remocao final:", constant_feature_columns)


Colunas removidas por 100% nulas: ['ABER-CKGL', 'ABER-CKP', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP']
Colunas removidas por serem labels/apoio: ['class', 'state']
Colunas constantes ainda presentes antes da remocao final: ['ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-PDG', 'QGL', 'T-PDG']


In [10]:
#imputacao temporal so para sinais analogicos
analog_df = feature_df[analog_columns].copy()
#para sinais de estado, usa propagacao simples dos ultimos valores observados
state_df = feature_df[state_columns].copy()

analog_df = analog_df.interpolate(method="time").ffill().bfill()
state_df = state_df.ffill().bfill()

#reconstroi o dataframe limpo juntando os dois grupos
clean_df = pd.concat([analog_df, state_df], axis=1).sort_index()

#remove colunas que continuaram constantes mesmo apos a limpeza
constant_after_clean = [col for col in clean_df.columns if clean_df[col].nunique(dropna=True) <= 1]
clean_df = clean_df.drop(columns=constant_after_clean, errors="ignore")

remaining_nulls = int(clean_df.isna().sum().sum())

print(f"Shape apos limpeza: {clean_df.shape}")
print(f"Valores nulos restantes: {remaining_nulls}")
print("Colunas constantes removidas apos limpeza:", constant_after_clean)


Shape apos limpeza: (21474, 6)
Valores nulos restantes: 0
Colunas constantes removidas apos limpeza: ['P-PDG', 'QGL', 'T-PDG', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO']


A expectativa aqui e que os nulos residuais caiam drasticamente em relacao ao notebook anterior. Se ainda restarem faltantes, isso normalmente indica uma coluna que nao deveria participar do modelo ou um trecho da serie em que a imputacao simples nao e suficiente.


In [11]:
#confere se ainda existem colunas problematicas apos a limpeza
clean_null_summary = (
    clean_df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("null_pct")
    .to_frame()
)
clean_null_summary.head(15)


,null_pct
P-ANULAR,0.0
P-JUS-CKGL,0.0
P-MON-CKP,0.0
P-TPT,0.0
T-JUS-CKP,0.0
T-TPT,0.0


## 5. Divisao temporal

A divisao respeita a ordem cronologica. Isso evita vazamento de informacao do futuro para o passado, o que e especialmente importante em series temporais.


In [12]:
#faz a divisao temporal preservando a ordem cronologica
n_samples = len(clean_df)
train_end = int(n_samples * 0.70)
val_end = train_end + int(n_samples * 0.15)

train_df = clean_df.iloc[:train_end].copy()
val_df = clean_df.iloc[train_end:val_end].copy()
test_df = clean_df.iloc[val_end:].copy()

split_summary = pd.DataFrame(
    {
        "dataset": ["train", "validation", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "start": [train_df.index.min(), val_df.index.min(), test_df.index.min()],
        "end": [train_df.index.max(), val_df.index.max(), test_df.index.max()],
    }
)
split_summary


,dataset,rows,start,end
0,train,15031,2017-02-01 01:02:07,2017-02-01 05:12:37
1,validation,3221,2017-02-01 05:12:38,2017-02-01 06:06:18
2,test,3222,2017-02-01 06:06:19,2017-02-01 07:00:00


## 6. Normalizacao e padronizacao

Nem todo problema responde melhor ao mesmo tipo de escalonamento. Por isso, o notebook gera tres alternativas:

- `MinMaxScaler`: util quando o modelo se beneficia de valores entre 0 e 1.
- `StandardScaler`: centraliza e padroniza por desvio padrao.
- `RobustScaler`: reduz o impacto de outliers.

O ajuste (`fit`) e feito somente no conjunto de treino. Validacao e teste recebem apenas `transform`, evitando vazamento de informacao.


In [13]:
#gera tres alternativas de escalonamento para comparacao posterior
scalers = {
    "minmax": MinMaxScaler(),
    "standard": StandardScaler(),
    "robust": RobustScaler(),
}

scaled_data = {}

for scaler_name, scaler in scalers.items():
    #o fit acontece apenas no treino para evitar vazamento de informacao
    train_scaled = pd.DataFrame(
        scaler.fit_transform(train_df),
        index=train_df.index,
        columns=train_df.columns,
    )
    val_scaled = pd.DataFrame(
        scaler.transform(val_df),
        index=val_df.index,
        columns=val_df.columns,
    )
    test_scaled = pd.DataFrame(
        scaler.transform(test_df),
        index=test_df.index,
        columns=test_df.columns,
    )
    scaled_data[scaler_name] = {
        "train": train_scaled,
        "validation": val_scaled,
        "test": test_scaled,
    }

print("Scalers gerados:", list(scaled_data.keys()))
scaled_data["minmax"]["train"].head()


Scalers gerados: ['minmax', 'standard', 'robust']


,P-ANULAR,P-JUS-CKGL,P-MON-CKP,P-TPT,T-JUS-CKP,T-TPT
timestamp,,,,,,
2017-02-01 01:02:07,1.0,0.00000,0.505540,0.666731,0.721035,0.827645
2017-02-01 01:02:08,1.0,0.00000,0.510260,0.666731,0.718693,0.827645
2017-02-01 01:02:09,1.0,0.00000,0.514978,0.666731,0.716355,0.827645
2017-02-01 01:02:10,1.0,0.00000,0.519697,0.666731,0.714009,0.827645
2017-02-01 01:02:11,1.0,0.00033,0.524415,0.666731,0.711667,0.827645


In [14]:
#compara estatisticas basicas das versoes escaladas
comparison_df = pd.DataFrame(
    {
        "raw_mean": train_df.mean(),
        "raw_std": train_df.std(),
        "minmax_mean": scaled_data["minmax"]["train"].mean(),
        "standard_mean": scaled_data["standard"]["train"].mean(),
        "standard_std": scaled_data["standard"]["train"].std(),
        "robust_median": scaled_data["robust"]["train"].median(),
    }
)
comparison_df.head(15)


,raw_mean,raw_std,minmax_mean,standard_mean,standard_std,robust_median
P-ANULAR,1.267896e+07,44451.458838,0.443790,3.327936e-15,1.000033,0.0
P-JUS-CKGL,1.564936e+06,874.629558,0.499996,7.212545e-14,1.000033,0.0
P-MON-CKP,1.580486e+06,173082.940016,0.464967,-2.940307e-16,1.000033,0.0
P-TPT,1.006459e+07,20071.513298,0.583633,2.967914e-14,1.000033,0.0
T-JUS-CKP,8.432790e+01,0.410506,0.604225,-2.063320e-14,1.000033,0.0
T-TPT,1.190548e+02,0.032629,0.628764,-3.612323e-13,1.000033,0.0


## 7. Persistencia dos artefatos

As proximas etapas vao consumir os dados prontos em vez de repetir toda a limpeza. Por isso, salvamos:

- conjuntos limpos sem escala;
- conjuntos escalados;
- um arquivo JSON com metadados da limpeza.


In [15]:
PREPROCESSED_DIR = ARTIFACTS_DIR / "preprocessed"
PREPROCESSED_DIR.mkdir(exist_ok=True)

#salva os conjuntos limpos sem escala para futuras comparacoes
train_df.to_parquet(PREPROCESSED_DIR / "train_clean.parquet")
val_df.to_parquet(PREPROCESSED_DIR / "validation_clean.parquet")
test_df.to_parquet(PREPROCESSED_DIR / "test_clean.parquet")

#salva tambem as versoes escaladas para treino, validacao e teste
for scaler_name, split_map in scaled_data.items():
    for split_name, split_df in split_map.items():
        split_df.to_parquet(PREPROCESSED_DIR / f"{split_name}_{scaler_name}.parquet")

#registra as decisoes do pre-processamento em um arquivo leve e legivel
metadata = {
    "source_file": str(parquet_file),
    "dropped_all_null_columns": drop_all_null,
    "dropped_constant_columns": constant_after_clean,
    "state_columns": state_columns,
    "analog_columns": analog_columns,
    "final_feature_columns": clean_df.columns.tolist(),
    "rows": {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df),
    },
}

with (PREPROCESSED_DIR / "preprocessing_metadata.json").open("w") as fh:
    json.dump(metadata, fh, indent=2)

print(f"Artefatos salvos em: {PREPROCESSED_DIR}")


Artefatos salvos em: /Users/tiagoriosdarocha/Desktop/lstm-w3/artifacts/preprocessed


## 8. Conclusao

Este notebook faz um pre-processamento mais defensavel do que a versao inicial, porque primeiro elimina colunas inviaveis e depois trata cada tipo de sinal de forma coerente. O resultado esperado para as proximas etapas e um conjunto de features com menos ruido estrutural, menos nulos espurios e mais consistencia entre treino, validacao e teste. O notebook `3-treino-validacao.ipynb` parte desses artefatos para montar sequencias e treinar um modelo inicial.
